# 03 カメラとリアルタイム処理

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 解説（この文章）→ コード → 結果（画像）→ ✅ チェックポイント、の順で進みます。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

**目標**：`picamera2` でカメラの**ライブ映像**を 1 フレームずつ取り出し、OpenCV で処理する。静止画から「動く画像」へ。

> **👥 ここからはカメラを使います。2人1組で共有**しましょう（**操作役／観察役を交代**）。1台のカメラでも、色トラッキング等は**画面を見せ合いながら**十分楽しめます。

> **🎯 このノートのゴール**
>
> - [ ] `picamera2` でフレームを numpy 配列として取得できる
> - [ ] 取得 → OpenCV 処理 → 表示の**ループ**を書ける
> - [ ] 特定の色の物体を**追いかける**（重心を打つ）処理を動かせる

## 3-0. picamera2 の基本と大事な注意
第1回でセンサ値を 2 秒ごとに読んだのと同じ発想で、カメラからは**フレームを繰り返し取り出します**。1 フレームは 01・02 で見た**あの numpy 配列**そのものです。

> **🟥 `format='RGB888'` なのに OpenCV では BGR**：`picamera2` の `RGB888` 設定で得た配列は、**メモリ上の並びが OpenCV の BGR と一致**します。つまり `capture_array()` の結果は**そのまま** `cv2.imwrite` や `cv2.cvtColor(..., BGR2...)` に渡せます。色が反転して見えるときだけ `cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)` を挟みます。**毎年つまずく所**です。

## 3-1. 準備（カメラを開いたままにする）
このノートでは何度も撮るので、カメラを開いたまま使います。**最後に 3-9 の後始末セルを必ず実行**してください。

In [ ]:
import cv2, numpy as np, time
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'  # 日本語ラベルの豆腐対策（要 fonts-noto-cjk）
from IPython.display import display, clear_output
%matplotlib inline
from picamera2 import Picamera2
picam = Picamera2()
picam.configure(picam.create_preview_configuration(main={'size':(640,480),'format':'RGB888'}))
picam.start(); time.sleep(1)
print('カメラ開始')

## 3-2. 1 フレーム取得して処理
取得 → グレー → エッジ、を 1 枚で確認します。

In [ ]:
frame = picam.capture_array()
edges = cv2.Canny(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 80, 160)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title('原画')
plt.subplot(1,2,2); plt.imshow(edges, cmap='gray'); plt.axis('off'); plt.title('エッジ')
plt.show()

## 3-3. ライブ表示（パラパラ更新）
取得を**ループ**にすれば「動画のリアルタイム処理」になります。下のセルは取得→処理→表示を繰り返して**動く映像**にします。`N` 枚で自動停止します。（処理を `edges` から変えると、ライブの見え方が変わります）

In [ ]:
N = 40
for i in range(N):
    frame = picam.capture_array()
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    view  = cv2.Canny(gray, 80, 160)            # ← ここを変えると見え方が変わる
    clear_output(wait=True)
    plt.figure(figsize=(6,4)); plt.imshow(view, cmap='gray'); plt.axis('off')
    plt.title(f'live {i+1}/{N}'); plt.show()
print('ライブ終了')

## 3-4. 色で物体を追いかける（カラートラッキング）
01・02 の HSV マスクを**ライブ映像に**適用し、見つけた色の塊の**重心**に印を打てば、簡単な**物体トラッキング**です。
**色のはっきりした物**（緑のペン、青いフタ等）を手に持って動かしてみましょう。

> 追いたい色に合わせて `inRange` の H 範囲を変えます（青=100–130／黄=20–35 が目安）。まず**静止画でマスクが綺麗に出る範囲を見つけてから**ライブに移すと速いです。

In [ ]:
LOWER, UPPER = (35,80,80), (85,255,255)   # 緑。青なら100-130, 黄なら20-35
N = 40
for i in range(N):
    frame = picam.capture_array()
    hsv  = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER, UPPER)
    cnts,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    big = [c for c in cnts if cv2.contourArea(c) > 300]
    if big:
        c = max(big, key=cv2.contourArea); M = cv2.moments(c)
        cx, cy = int(M['m10']/M['m00']), int(M['m01']/M['m00'])
        cv2.circle(frame, (cx,cy), 12, (0,0,255), -1)
    clear_output(wait=True)
    plt.figure(figsize=(6,4)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'tracking {i+1}/{N}'); plt.show()
print('終了')

✅ **チェックポイント**：手に持った色の物に、赤い点が追従すれば成功です。

## 3-5. （発展）ブラウザにライブ配信したい人へ
SSH だけだと画面を開けないので、Pi の中に小さな Web サーバを立て、**1 フレームずつ JPEG にして送り続ける**（**MJPEG**）方式でブラウザに流せます。追加インストール不要（Python 標準ライブラリ＋OpenCV）。

この配信はノート内ではなく**スクリプト**で動かします（`~/dx-session02/notebooks/scripts_reference/cam_web.py` などを参照）。Pi のターミナルで実行し、自分の PC のブラウザで `http://<Pi の IP>:8000` を開くと、処理済み映像が動画で見られます。（`<Pi の IP>` は `hostname -I` の値。カクつくときは解像度を `(320,240)` に下げる）

## 3-9. 後始末（必ず実行）
カメラを解放します。次のノート（04 AIカメラ）に進む前に実行してください。

In [ ]:
picam.close(); print('カメラ解放')

## ✅ 到達確認

- [ ] `picamera2` でライブフレームを取得し OpenCV に渡せた
- [ ] エッジ／カラートラッキングをループで動かせた
- [ ] 後始末（`picam.close()`）を実行した